In [0]:
# Read the CSV file into a DataFrame

df_sales = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load("/Volumes/pyspark_real_time/source/source_data/sales/sales.csv")

df_sales.show(5)

In [0]:
df_prod = spark.read.format("json").option("header", "true").option("inferSchema", "true").load("/Volumes/pyspark_real_time/source/source_data/products/products.json")

df_prod.show(5)

##Spark streaming

In [0]:
entities = {
    "sales": "csv",
    "products": "json"
}

for entity, source_format in entities.items():

    source_path = f"/Volumes/pyspark_real_time/source/source_data/{entity}"
    checkpoint_path = f"/Volumes/pyspark_real_time/bronze/checkpoint/{entity}"
    table_name = f"pyspark_real_time.bronze.{entity}"

    reader = spark.read.format(source_format)

    if source_format == "csv":
        reader = reader.option("header", "true")

    entity_schema = reader.option("inferSchema", True).load(source_path).schema

    stream_reader = spark.readStream.format(source_format)

    if source_format == "csv":
        stream_reader = stream_reader.option("header", "true")

    df_batch = stream_reader.schema(entity_schema).load(source_path)

    query = (
        df_batch.writeStream
        .format("delta")
        .outputMode("append")
        .option("checkpointLocation", checkpoint_path)
        .trigger(once=True)
        .toTable(table_name)
    )

    query.awaitTermination()